Here is an example of building a dataset from an SDF file.

Dataset preprocessing may take minutes to hours depending on size.

In [ ]:
import os
import h5py
import pandas as pd
from rdkit import Chem


sdf_path = './data/TorsionNet.sdf'
output_dir = './data/TorsionNet.hdf5'


if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Read SDF
supplier = Chem.SDMolSupplier(sdf_path, removeHs=False)

data = {
    'hydrogenated_mapped_smiles': [],
    'coordinates': [],
    'Energy': [],
    'FirstLineDescription_1': [],
    'FirstLineDescription_2': [],
    'ScanVar_1': []
}


for mol in supplier:
    if mol is None:
        continue
    first_line = mol.GetProp("_Name")
    import re
    first_line_numbers = re.findall(r'\d+', first_line)

    first_part = first_line_numbers[0].zfill(3)[:3] 
    second_part = first_line_numbers[1].zfill(2)[:2]  

    data['FirstLineDescription_1'].append(int(first_part))
    data['FirstLineDescription_2'].append(int(second_part))

    # Get energy
    Energy = float(mol.GetProp('Energy')) if mol.HasProp('Energy') else None
    print('FirstLineDescription_1:',int(first_part),'FirstLineDescription_2:',int(second_part),'Energy:', Energy)

    # Get SMILES
    smiles = Chem.MolToSmiles(mol)
    
    # Get coordinates
    conf = mol.GetConformer()
    coords = []
    for atom in mol.GetAtoms():
        pos = conf.GetAtomPosition(atom.GetIdx())
        coords.append([pos.x, pos.y, pos.z])
    data['coordinates'].append(coords)
   
    mol_h = Chem.AddHs(mol)
    for atom in mol_h.GetAtoms():
        atom.SetAtomMapNum(atom.GetIdx() + 1)

    mapped_smiles = Chem.MolToSmiles(mol_h)
    data['hydrogenated_mapped_smiles'].append(mapped_smiles)

    ScanVar_1 = float(mol.GetProp('Torsion_angle')) if mol.HasProp('Torsion_angle') else None
    data['Energy'].append(Energy)
    data['ScanVar_1'].append(ScanVar_1)

# DataFrame 
df = pd.DataFrame(data)
grouped = df.groupby('hydrogenated_mapped_smiles')

hdf5_path=f"{output_dir}/Torsionnet_name.hdf5"
with h5py.File(hdf5_path, 'w') as hdf5_file:
    for smiles, group in grouped:
        group_data = group.to_dict(orient='list')
        group_path = smiles.replace('/', '_')
        
        # Create group
        mol_group = hdf5_file.create_group(group_path)
        
        mol_group.create_dataset('hydrogenated_mapped_smiles', data=group_data['hydrogenated_mapped_smiles'])
        mol_group.create_dataset('coordinates', data=group_data['coordinates'])
        mol_group.create_dataset('Energy', data=group_data['Energy'])
        mol_group.create_dataset('ScanVar_1', data=group_data['ScanVar_1'])
        mol_group.create_dataset('FirstLineDescription_1', data=group_data['FirstLineDescription_1'])
        mol_group.create_dataset('FirstLineDescription_2', data=group_data['FirstLineDescription_2'])

print("Save all groups into HDF5.")


FirstLineDescription_1: 1 FirstLineDescription_2: 1 Energy: -351878.59773685894
FirstLineDescription_1: 1 FirstLineDescription_2: 2 Energy: -351878.8872714926
FirstLineDescription_1: 1 FirstLineDescription_2: 3 Energy: -351879.1376794793
FirstLineDescription_1: 1 FirstLineDescription_2: 4 Energy: -351879.4502502915
FirstLineDescription_1: 1 FirstLineDescription_2: 5 Energy: -351879.98820933764
FirstLineDescription_1: 1 FirstLineDescription_2: 6 Energy: -351881.02793301374
FirstLineDescription_1: 1 FirstLineDescription_2: 7 Energy: -351882.81688121293
FirstLineDescription_1: 1 FirstLineDescription_2: 8 Energy: -351885.08982777025
FirstLineDescription_1: 1 FirstLineDescription_2: 9 Energy: -351887.1192452556
FirstLineDescription_1: 1 FirstLineDescription_2: 10 Energy: -351888.57099454396
FirstLineDescription_1: 1 FirstLineDescription_2: 11 Energy: -351889.40821476415
FirstLineDescription_1: 1 FirstLineDescription_2: 12 Energy: -351889.6788364206
FirstLineDescription_1: 1 FirstLineDescrip

[03:12:07] Warning: molecule is tagged as 3D, but all Z coords are zero
[03:12:07] Warning: molecule is tagged as 3D, but all Z coords are zero


FirstLineDescription_1: 197 FirstLineDescription_2: 6 Energy: -286491.30341618776
FirstLineDescription_1: 197 FirstLineDescription_2: 7 Energy: -286490.94572111964
FirstLineDescription_1: 197 FirstLineDescription_2: 8 Energy: -286492.100437451
FirstLineDescription_1: 197 FirstLineDescription_2: 9 Energy: -286492.6243370818
FirstLineDescription_1: 197 FirstLineDescription_2: 10 Energy: -286492.53405947506
FirstLineDescription_1: 197 FirstLineDescription_2: 11 Energy: -286492.05971031665
FirstLineDescription_1: 197 FirstLineDescription_2: 12 Energy: -286491.4941066576
FirstLineDescription_1: 197 FirstLineDescription_2: 13 Energy: -286492.0597223525
FirstLineDescription_1: 197 FirstLineDescription_2: 14 Energy: -286492.53400869673
FirstLineDescription_1: 197 FirstLineDescription_2: 15 Energy: -286492.62461913994
FirstLineDescription_1: 197 FirstLineDescription_2: 16 Energy: -286492.1002796917
FirstLineDescription_1: 197 FirstLineDescription_2: 17 Energy: -286490.93952221435
FirstLineDescr

[03:12:09] Warning: molecule is tagged as 3D, but all Z coords are zero


FirstLineDescription_1: 317 FirstLineDescription_2: 9 Energy: -490946.7884828439
FirstLineDescription_1: 317 FirstLineDescription_2: 10 Energy: -490947.37771065865
FirstLineDescription_1: 317 FirstLineDescription_2: 11 Energy: -490947.5742736984
FirstLineDescription_1: 317 FirstLineDescription_2: 12 Energy: -490949.0390663693
FirstLineDescription_1: 317 FirstLineDescription_2: 13 Energy: -490948.86951375403
FirstLineDescription_1: 317 FirstLineDescription_2: 14 Energy: -490948.3725706011
FirstLineDescription_1: 317 FirstLineDescription_2: 15 Energy: -490947.5273054654
FirstLineDescription_1: 317 FirstLineDescription_2: 16 Energy: -490946.3723219979
FirstLineDescription_1: 317 FirstLineDescription_2: 17 Energy: -490945.2274870402
FirstLineDescription_1: 317 FirstLineDescription_2: 18 Energy: -490944.71895445156
FirstLineDescription_1: 317 FirstLineDescription_2: 19 Energy: -490945.155807316
FirstLineDescription_1: 317 FirstLineDescription_2: 20 Energy: -490946.1961713393
FirstLineDescri

[03:12:11] Warning: molecule is tagged as 3D, but all Z coords are zero
[03:12:11] Warning: molecule is tagged as 3D, but all Z coords are zero


FirstLineDescription_1: 458 FirstLineDescription_2: 3 Energy: -608571.698006842
FirstLineDescription_1: 458 FirstLineDescription_2: 4 Energy: -608571.3717274489
FirstLineDescription_1: 458 FirstLineDescription_2: 5 Energy: -608570.7483986311
FirstLineDescription_1: 458 FirstLineDescription_2: 6 Energy: -608570.3652075765
FirstLineDescription_1: 458 FirstLineDescription_2: 7 Energy: -608570.7317229211
FirstLineDescription_1: 458 FirstLineDescription_2: 8 Energy: -608571.3862563785
FirstLineDescription_1: 458 FirstLineDescription_2: 9 Energy: -608571.6946497349
FirstLineDescription_1: 458 FirstLineDescription_2: 10 Energy: -608571.0175156947
FirstLineDescription_1: 458 FirstLineDescription_2: 11 Energy: -608569.4095159954
FirstLineDescription_1: 458 FirstLineDescription_2: 12 Energy: -608567.5734907615
FirstLineDescription_1: 458 FirstLineDescription_2: 13 Energy: -608569.4095079078
FirstLineDescription_1: 458 FirstLineDescription_2: 14 Energy: -608571.0175380259
FirstLineDescription_1: 

[03:12:12] Warning: molecule is tagged as 3D, but all Z coords are zero


Save all groups into HDF5.


In [ ]:
import os, sys
import numpy as np
import h5py
import torch
import resff
from resff.units import *
import pandas as pd
from openff.toolkit.topology import Molecule
from simtk import unit
from simtk.unit import Quantity


path = '/home/datahouse1/jiangxinyu/ResFF-github/data/raw_data/Torsionnet_name.hdf5'
idx = 0

with h5py.File(path, "r") as h5file:
    # Initialize an empty list to store the data for all molecules
    # Iterate through each key (corresponding to the molecule's SMILES)
    for smiles_key in h5file.keys():
        # Access the group corresponding to the current molecule
        record = h5file[smiles_key]
        # Extract properties for the current molecule (group)
        print(smiles_key)
        smi = record["hydrogenated_mapped_smiles"][0].decode('UTF-8')
          # Skip the loop if the SMILES contains 'Li', 'Na', 'Mg', 'K', or 'Ca'
        if any(element in smi for element in ['Li', 'Na', 'Mg', 'K', 'Ca']):
            print(f"Skipping SMILES: {smi}")
            continue

        print(smi)
        offmol = Molecule.from_mapped_smiles(smi, allow_undefined_stereo=True)
    
        try:
            g = resff.Graph(offmol)
        except:
        #    failures = open('failures.txt', 'a')
        #    failures.write("{}\t{}\t{}\n".format(smiles_key, smi, idx))
        #    failures.close()
            continue

        idx += 1
        energy = record["Energy"][()]
        xyz = record["coordinates"][()].transpose(1, 0, 2)
        ScanVar_1 = record['ScanVar_1'][()]
        FirstLineDescription_1 = record['FirstLineDescription_1'][()]
        FirstLineDescription_2 = record['FirstLineDescription_2'][()]

        g.nodes['g'].data['ScanVar_1'] = torch.tensor(
                ScanVar_1,
            dtype=torch.get_default_dtype(),
        )[None, :]

        g.nodes['g'].data['FirstLineDescription_1'] = torch.tensor(
                FirstLineDescription_1,
            dtype=torch.get_default_dtype(),
        )[None, :]

        g.nodes['g'].data['FirstLineDescription_2'] = torch.tensor(
                FirstLineDescription_2,
            dtype=torch.get_default_dtype(),
        )[None, :]

        g.nodes["g"].data["u_ref"] = torch.tensor(
                Quantity(
                    energy,
                    resff.units.HARTREE_PER_PARTICLE,
                ).value_in_unit(resff.units.ENERGY_UNIT),
            dtype=torch.get_default_dtype(),
        )[None, :]

        g.nodes["n1"].data["xyz"] = torch.tensor(
            Quantity(
                xyz,
                unit.bohr,
            ).value_in_unit(resff.units.DISTANCE_UNIT),
            requires_grad=True,
            dtype=torch.get_default_dtype(),
        )
    
        print('finish')
    
        gs = []
        gs.append(g)

        ds = resff.data.dataset.GraphDataset(gs)
        for g in ds :  
            save_dir = "ResFF/data/Torsion_Net/%s"%idx
            resff.Graph.save(g,save_dir)
    
